# Self-Verification / CoVe

Retain the initial response and a separate verifier response; score the verifier.

This executed notebook uses only public prompts, responses, and aggregate analysis. It does not embed answer keys or row-level correctness labels.

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

root = Path.cwd().resolve()
manifest_path = root / 'tier1_manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}

def read_public_csv(path):
    if not path.exists():
        return pd.DataFrame()
    frame = pd.read_csv(path, keep_default_na=False)
    forbidden = {'gold', 'ground_truth', 'correct_answer', 'normalized_gold', 'is_correct'}
    exposed = forbidden.intersection(str(column).lower() for column in frame.columns)
    if exposed:
        raise ValueError(f'public analysis contains forbidden row-level columns: {sorted(exposed)}')
    return frame

print('model:', manifest.get('model', 'not prepared'))
print('reasoning effort:', manifest.get('reasoning_effort', 'not prepared'))
print('prompt version:', manifest.get('prompt_version', 'not prepared'))


model: gpt-5.6-sol
reasoning effort: high
prompt version: gpt56-tier1-v1


In [2]:
condition = 'self_verification_cove'
queues = ('self_verification_cove_initial', 'self_verification_cove_verification')
coverage = []
result_frames = []
for queue in queues:
    dataset_path = root / 'Dataset' / f'{queue}.csv'
    result_path = root / 'Result' / f'{queue}.csv'
    dataset = read_public_csv(dataset_path)
    result = read_public_csv(result_path)
    answered = 0
    if not result.empty and 'model_response' in result:
        answered = int(result['model_response'].astype(str).str.strip().ne('').sum())
        result_frames.append(result.assign(source_queue=queue))
    coverage.append({
        'queue': queue,
        'dataset_rows': len(dataset),
        'result_rows': len(result),
        'answered_rows': answered,
        'unresolved_rows': max(len(result) - answered, 0),
    })

display(pd.DataFrame(coverage))
if result_frames:
    results = pd.concat(result_frames, ignore_index=True)
    group_columns = [column for column in ('task', 'variant', 'stage', 'path_index') if column in results]
    display(
        results.groupby(group_columns, dropna=False)
        .agg(responses=('response_id', 'count'), answered=('model_response', lambda values: values.astype(str).str.strip().ne('').sum()))
        .reset_index()
    )
else:
    print('Result files are not available yet.')

analysis_dir = root / 'Analysis'
shown = 0
if analysis_dir.exists():
    for path in sorted(analysis_dir.glob('*.csv')):
        frame = read_public_csv(path)
        if frame.empty:
            continue
        selector = None
        if 'condition' in frame.columns:
            selector = frame['condition'].astype(str).eq(condition)
        elif 'method' in frame.columns:
            selector = frame['method'].astype(str).eq(condition)
        if selector is not None and bool(selector.any()):
            print(path.name)
            selected = frame.loc[selector].reset_index(drop=True)
            if 'evaluation_id' in selected.columns and len(selected) > 100:
                print('public row-level aggregate rows:', len(selected))
                grouping = [column for column in ('task', 'variant', 'agreement', 'status', 'answer_changed') if column in selected]
                if grouping:
                    display(selected.groupby(grouping, dropna=False).size().rename('rows').reset_index())
            else:
                display(selected)
            shown += 1
if not shown:
    print('No condition-level public analysis tables are available yet.')


,queue,dataset_rows,result_rows,answered_rows,unresolved_rows
0,self_verification_cove_initial,450,450,450,0
1,self_verification_cove_verification,450,450,450,0


,task,variant,stage,path_index,responses,answered
0,obfus_blood_relation,base,initial,,50,50
1,obfus_blood_relation,base,verification,,50,50
2,obfus_blood_relation,obfuscation_l1,initial,,50,50
3,obfus_blood_relation,obfuscation_l1,verification,,50,50
4,obfus_blood_relation,obfuscation_l2,initial,,50,50
5,obfus_blood_relation,obfuscation_l2,verification,,50,50
6,obfus_direction_sense,base,initial,,50,50
7,obfus_direction_sense,base,verification,,50,50
8,obfus_direction_sense,obfuscation,initial,,50,50
9,obfus_direction_sense,obfuscation,verification,,50,50


accuracy_by_cell.csv


,model,condition,task,task_title,variant,correct,total,accuracy_pct,response_or_consensus_failures
0,gpt-5.6-sol,self_verification_cove,obfus_fol,FOL,base,41,50,82.0,0
1,gpt-5.6-sol,self_verification_cove,obfus_fol,FOL,obfuscation,32,50,64.0,0
2,gpt-5.6-sol,self_verification_cove,obfus_blood_relation,Blood Relation,base,38,50,76.0,0
3,gpt-5.6-sol,self_verification_cove,obfus_blood_relation,Blood Relation,obfuscation_l1,35,50,70.0,0
4,gpt-5.6-sol,self_verification_cove,obfus_blood_relation,Blood Relation,obfuscation_l2,36,50,72.0,0
5,gpt-5.6-sol,self_verification_cove,obfus_number_series,Number Series,base,40,50,80.0,0
6,gpt-5.6-sol,self_verification_cove,obfus_number_series,Number Series,obfuscation,40,50,80.0,0
7,gpt-5.6-sol,self_verification_cove,obfus_direction_sense,Direction Sense,base,24,50,48.0,0
8,gpt-5.6-sol,self_verification_cove,obfus_direction_sense,Direction Sense,obfuscation,20,50,40.0,0


base_obfuscated_gaps.csv


...

,task,variant,status,answer_changed,rows
0,obfus_blood_relation,base,answered,False,50
1,obfus_blood_relation,obfuscation_l1,answered,False,50
2,obfus_blood_relation,obfuscation_l2,answered,False,50
3,obfus_direction_sense,base,answered,False,50
4,obfus_direction_sense,obfuscation,answered,False,48
5,obfus_direction_sense,obfuscation,answered,True,2
6,obfus_fol,base,answered,False,49
7,obfus_fol,base,answered,True,1
8,obfus_fol,obfuscation,answered,False,47
9,obfus_fol,obfuscation,answered,True,3


improvement_over_control.csv


,condition,task,variant,method_correct,control_correct,total,method_accuracy_pct,control_accuracy_pct,improvement_pp
0,self_verification_cove,obfus_fol,obfuscation,32,30,50,64.0,60.0,4.0
1,self_verification_cove,obfus_blood_relation,obfuscation_l1,35,36,50,70.0,72.0,-2.0
2,self_verification_cove,obfus_blood_relation,obfuscation_l2,36,37,50,72.0,74.0,-2.0
3,self_verification_cove,obfus_number_series,obfuscation,40,42,50,80.0,84.0,-4.0
4,self_verification_cove,obfus_direction_sense,obfuscation,20,18,50,40.0,36.0,4.0


paired_retention.csv


,condition,task,variant,both_correct,base_only_correct,obfuscated_only_correct,both_wrong,total,paired_retention_pct
0,self_verification_cove,obfus_fol,obfuscation,28,13,4,5,50,68.2927
1,self_verification_cove,obfus_blood_relation,obfuscation_l1,35,3,0,12,50,92.1053
2,self_verification_cove,obfus_blood_relation,obfuscation_l2,36,2,0,12,50,94.7368
3,self_verification_cove,obfus_number_series,obfuscation,40,0,0,10,50,100.0000
4,self_verification_cove,obfus_direction_sense,obfuscation,20,4,0,26,50,83.3333
